# 1. Train the vehicle detector on UVH-26 (Colab GPU)

Fine-tunes YOLO-family detectors on the project dataset (UVH-26, 14 Indian vehicle classes) and benchmarks
them so the final detector is *selected after benchmarking*, as the presentation proposes.

**Before running:** Runtime → Change runtime type → T4 GPU. Your dataset must be on Google Drive at
`MyDrive/UVH26_12K_YOLO` with `images/{train,val,test}` and matching `labels/{train,val,test}` folders
(YOLO txt format), plus the `data.yaml` already in the project.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install ultralytics

In [ ]:
import os, yaml
DATA = '/content/drive/MyDrive/UVH26_12K_YOLO'
OUT = '/content/drive/MyDrive/traffic_ai_runs'
os.makedirs(OUT, exist_ok=True)
cfg = {'path': DATA, 'train': 'images/train', 'val': 'images/val', 'test': 'images/test', 'nc': 14,
       'names': ['Hatchback','Sedan','SUV','MUV','Bus','Truck','Three-wheeler','Two-wheeler','LCV',
                 'Mini-bus','Tempo-traveller','Bicycle','Van','Others']}
yaml.safe_dump(cfg, open('/content/uvh26.yaml', 'w'), sort_keys=False)
for split in ['train', 'val', 'test']:
    n_img = len(os.listdir(f'{DATA}/images/{split}'))
    n_lbl = len(os.listdir(f'{DATA}/labels/{split}')) if os.path.isdir(f'{DATA}/labels/{split}') else 0
    print(f'{split}: {n_img} images, {n_lbl} label files')

## Train candidates
Each run takes roughly 1–3 hours on a T4 at 60 epochs. Both `yolo26s` (YOLO family) and `rtdetr-l` (real-time detection transformer) are trained with the same data and image size so the comparison is fair.
RT-DETR is slower to train (expect 2–3× the YOLO time); if Colab disconnects, re-run the cell — `exist_ok=True` keeps the run folder and you can pass `resume=True`.

In [ ]:
from ultralytics import YOLO, RTDETR
CANDIDATES = ['yolo26s.pt', 'rtdetr-l.pt']   # YOLO-family vs transformer (add 'yolo11s.pt' if time allows)
EPOCHS = 60
for w in CANDIDATES:
    m = RTDETR(w) if 'rtdetr' in w else YOLO(w)
    # RT-DETR needs a smaller batch and lower learning rate on a T4 (16 GB)
    extra = dict(batch=4, lr0=1e-4, optimizer='AdamW', amp=True) if 'rtdetr' in w else dict(batch=16)
    m.train(data='/content/uvh26.yaml', epochs=EPOCHS, imgsz=960, patience=15,
            project=OUT, name=w.replace('.pt', '_uvh26'), exist_ok=True,
            mosaic=1.0, close_mosaic=10, cos_lr=True, **extra)

## Benchmark: accuracy and speed on the test split

In [ ]:
import pandas as pd, time, glob, numpy as np, cv2
rows = []
test_imgs = sorted(glob.glob(f'{DATA}/images/test/*.jpg'))[:200]
for w in CANDIDATES:
    best = f"{OUT}/{w.replace('.pt', '_uvh26')}/weights/best.pt"
    m = RTDETR(best) if 'rtdetr' in w else YOLO(best)
    r = m.val(data='/content/uvh26.yaml', split='test', imgsz=960, verbose=False)
    imgs = [cv2.imread(p) for p in test_imgs[:100]]
    m.predict(imgs[:5], imgsz=960, verbose=False)  # warm-up
    t0 = time.time()
    for im in imgs:
        m.predict(im, imgsz=960, verbose=False)
    fps = len(imgs) / (time.time() - t0)
    rows.append({'model': w, 'mAP50': r.box.map50, 'mAP50-95': r.box.map, 'precision': r.box.mp,
                 'recall': r.box.mr, 'FPS (T4, 960px)': fps})
bench = pd.DataFrame(rows).round(3)
bench.to_csv(f'{OUT}/detector_benchmark.csv', index=False)
bench

In [ ]:
# Per-class AP for both models side by side (copy the winner's best.pt into outputs/weights/)
per_class = {}
for w in CANDIDATES:
    best = f"{OUT}/{w.replace('.pt', '_uvh26')}/weights/best.pt"
    mm = RTDETR(best) if 'rtdetr' in w else YOLO(best)
    r = mm.val(data='/content/uvh26.yaml', split='test', imgsz=960, verbose=False)
    per_class[w] = r.box.maps
    if w == CANDIDATES[0]:
        m = mm
pc = pd.DataFrame(per_class, index=list(m.names.values())).round(3)
pc.to_csv(f'{OUT}/detector_per_class_ap.csv')
pc

## Robustness slices (day / night / rain)
If you tag test images by condition (e.g. a CSV with `image,condition`), evaluate each slice separately by
writing one YAML per slice whose `test:` entry is a txt file listing those images.

In [ ]:
import os
cond_csv = f'{DATA}/test_conditions.csv'   # optional: columns image,condition
if os.path.exists(cond_csv):
    c = pd.read_csv(cond_csv)
    for cond, g in c.groupby('condition'):
        lst = f'/content/test_{cond}.txt'
        open(lst, 'w').write('\n'.join(f'{DATA}/images/test/{i}' for i in g.image))
        y = dict(cfg, test=lst); yaml.safe_dump(y, open(f'/content/uvh26_{cond}.yaml', 'w'))
        r = m.val(data=f'/content/uvh26_{cond}.yaml', split='test', imgsz=960, verbose=False)
        print(f'{cond:8s} mAP50={r.box.map50:.3f}  mAP50-95={r.box.map:.3f}')
else:
    print('No test_conditions.csv found; skipping condition slices.')

## Tracking check and export
Run ByteTrack on any traffic video to check identity stability, then use the weights locally:

```
python -m traffic_ai.cli video --source junction.mp4 --camera configs/camera_example.json --model best.pt
```

In [ ]:
# Optional: upload a short junction video to Drive and track it
VIDEO = '/content/drive/MyDrive/junction.mp4'
if os.path.exists(VIDEO):
    m.track(VIDEO, tracker='bytetrack.yaml', imgsz=960, conf=0.3, save=True, project=OUT, name='track_demo')
m.export(format='onnx', imgsz=960)  # for edge devices (Jetson / OpenVINO)